%md ## 0 · Imports & Logging

In [0]:
import json
import re
import logging
from datetime import datetime, timezone

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, StringType, LongType
from pyspark.sql.window import Window

logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("sentiment_analysis")

spark = SparkSession.builder.appName("youtube_sentiment").getOrCreate()

%md ## 1 · Pipeline Parameters

In [0]:
dbutils.widgets.text("bucket_name",    "youtube-capstone-bucket-team6", "S3 Bucket Name")
dbutils.widgets.text("catalog_name",   "youtube-capstone-catalog",      "Unity Catalog Name")
dbutils.widgets.text("run_date",       datetime.now(timezone.utc).strftime("%Y-%m-%d"), "Run Date")
dbutils.widgets.text("top_keywords",   "20",  "Top N keywords per sentiment bucket")
dbutils.widgets.text("min_word_len",   "3",   "Min word length for keyword extraction")

BUCKET_NAME   = dbutils.widgets.get("bucket_name")
CATALOG_NAME  = dbutils.widgets.get("catalog_name")
RUN_DATE      = dbutils.widgets.get("run_date")
TOP_KEYWORDS  = int(dbutils.widgets.get("top_keywords"))
MIN_WORD_LEN  = int(dbutils.widgets.get("min_word_len"))

BASE_PATH     = f"s3://{BUCKET_NAME}/youtube_pipeline"
SILVER_PATH   = f"{BASE_PATH}/silver"
SENTIMENT_PATH     = f"{BASE_PATH}/sentiment_analysis"


SENTIMENT_DB  = f"`{CATALOG_NAME}`.sentiment_analysis"

logger.info(f"Sentiment config → catalog={CATALOG_NAME}, run_date={RUN_DATE}")
logger.info(f"Silver path → {SILVER_PATH}")
logger.info(f"Gold path   → {SENTIMENT_PATH}")

%md ## 2 · Load Silver Tables

In [0]:
df_silver_comments = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/silver_comments")
    .filter(F.col("run_date") == RUN_DATE)
)

df_silver_videos = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/silver_videos")
    .filter(F.col("run_date") == RUN_DATE)
)

logger.info(f"Silver comments loaded : {df_silver_comments.count():,} rows")
logger.info(f"Silver videos loaded   : {df_silver_videos.count():,} rows")

df_silver_comments.printSchema()

root
 |-- comment_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- like_count: long (nullable = true)
 |-- comment_date: date (nullable = true)
 |-- run_date: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- comment_ts: timestamp (nullable = true)
 |-- comment_year: integer (nullable = true)
 |-- comment_month: integer (nullable = true)
 |-- word_count: integer (nullable = true)
 |-- char_count: integer (nullable = true)
 |-- has_emoji: boolean (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)
 |-- _data_layer: string (nullable = true)



%md ## 3 · Sentiment Word Lists

These word lists **match the Snowflake UDF exactly** so scores are
consistent whether computed in Databricks or Snowflake.

In [0]:
#Positive words (mirrors Snowflake UDF POS_WORDS) 
POS_WORDS = {
    'good','great','excellent','amazing','awesome','fantastic','wonderful',
    'outstanding','brilliant','superb','love','loved','like','liked','enjoy',
    'enjoyed','perfect','best','beautiful','incredible','impressive','helpful',
    'informative','educational','insightful','clear','interesting','useful',
    'valuable','recommend','recommended','happy','glad','excited','grateful',
    'thankful','appreciate','appreciated','nice','well','better','improved',
    'innovative','creative','quality','professional','efficient','effective',
    'easy','simple','fun','entertaining','engaging','motivating','inspiring',
    'top','exceptional','positive','success','successful','win','winner',
    'legendary','iconic','masterpiece','gem','treasure','phenomenal','stellar'
}

# Negative words (mirrors Snowflake UDF NEG_WORDS) 
NEG_WORDS = {
    'bad','terrible','awful','horrible','disgusting','hate','hated','dislike',
    'disliked','boring','dull','worst','poor','disappointing','disappointed',
    'mediocre','annoying','annoyed','frustrating','frustrated','confusing',
    'confused','useless','waste','wasted','overrated','misleading','wrong',
    'incorrect','false','fake','spam','scam','clickbait','pathetic','failure',
    'failed','broken','buggy','slow','difficult','complicated','hard','worse',
    'negative','problem','issue','error','mistake','inaccurate','irrelevant',
    'repetitive','shallow','toxic','rude','offensive','inappropriate',
    'stupid','dumb','worthless','pointless','trash','garbage','cringe','ugly'
}

# Intensifiers → multiply score by 1.5 
INTENSIFIERS = {
    'very','really','so','extremely','absolutely','totally','completely',
    'incredibly','seriously','highly','super','exceptionally','utterly'
}

# Negation words → flip score sign
NEGATIONS = {
    'not','no','never','neither','nothing','nobody','nowhere','nor',
    "n't","cannot","cant","dont","doesnt","isnt","wasnt","wouldnt","shouldnt"
}

# Emotion classifier word lists (mirrors Snowflake udf_emotion) 
EMOTION_MAP = {
    'JOY'     : ['happy','joy','excited','love','amazing','wonderful','yay','great','laugh','fun'],
    'ANGER'   : ['angry','furious','rage','annoying','hate','frustrated','outrage','mad'],
    'SADNESS' : ['sad','unhappy','depressed','sorry','miss','disappoint','unfortunate'],
    'FEAR'    : ['scared','afraid','worried','anxious','terrified','concern','panic'],
    'SURPRISE': ['wow','omg','shocked','surprising','unexpected','unbelievable','astonishing'],
    'DISGUST' : ['disgusting','gross','horrible','awful','revolting','nasty','offensive','trash'],
}

# Stopwords for keyword extraction
STOPWORDS = {
    'this','that','with','have','from','they','will','been','were','what',
    'when','where','which','there','their','about','would','could','should',
    'just','like','your','more','also','some','than','then','into','over',
    'only','very','even','most','such','after','before','because','through',
    'those','these','being','other','each','both','much','many','same',
    'here','well','back','still','make','made','know','think','come',
    'good','great',  
}

logger.info(f"Loaded {len(POS_WORDS)} positive words, {len(NEG_WORDS)} negative words")

%md ## 4 · Sentiment Score UDF

Python implementation of the Snowflake `udf_sentiment_score` JS function.
Handles: negation (prev 1-2 words), intensifiers (prev 1-2 words → ×1.5)

In [0]:
_pos   = POS_WORDS
_neg   = NEG_WORDS
_int   = INTENSIFIERS
_negs  = NEGATIONS


@F.udf(returnType=FloatType())
def udf_sentiment_score(text: str) -> float:

    if not text or not text.strip():
        return 0.0

    pos_words  = _pos
    neg_words  = _neg
    intensifiers = _int
    negations  = _negs

    tokens = re.sub(r"[^a-z0-9'\s]", " ", text.lower()).split()
    tokens = [w for w in tokens if w]

    score      = 0.0
    word_count = 0

    for i, word in enumerate(tokens):
        prev1 = tokens[i - 1] if i > 0 else ""
        prev2 = tokens[i - 2] if i > 1 else ""

        negated     = (prev1 in negations) or (prev2 in negations)
        intensified = (prev1 in intensifiers) or (prev2 in intensifiers)
        multiplier  = 1.5 if intensified else 1.0

        if word in pos_words:
            score      += (-1.0 * multiplier) if negated else (+1.0 * multiplier)
            word_count += 1
        elif word in neg_words:
            score      += (+1.0 * multiplier) if negated else (-1.0 * multiplier)
            word_count += 1

    if word_count == 0:
        return 0.0

    raw = score / word_count
    return float(max(-1.0, min(1.0, raw)))


@F.udf(returnType=StringType())
def udf_sentiment_label(score: float) -> str:
    if score is None:
        return "NEUTRAL"
    if score > 0.15:
        return "POSITIVE"
    elif score < -0.15:
        return "NEGATIVE"
    return "NEUTRAL"


_emotion_map = EMOTION_MAP


@F.udf(returnType=StringType())
def udf_emotion(text: str) -> str:
    if not text:
        return "NEUTRAL"

    t      = text.lower()
    e_map  = _emotion_map
    best   = "NEUTRAL"
    best_n = 0

    for emotion, words in e_map.items():
        count = sum(1 for w in words if w in t)
        if count > best_n:
            best_n = count
            best   = emotion

    return best


logger.info("Sentiment UDFs registered")

%md ## 5 · Gold Table Writer Helper

In [0]:
def write_sentiment_table(df: DataFrame, table_name: str, partition_cols: list = None) -> int:
    """Write Gold Delta table and register in Unity Catalog. Returns row count."""
    path = f"{SENTIMENT_PATH}/{table_name}"

    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("delta.enableChangeDataFeed", "true")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SENTIMENT_DB}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SENTIMENT_DB}.{table_name}
        USING DELTA
        LOCATION '{path}'
    """)
    spark.sql(f"OPTIMIZE {SENTIMENT_DB}.{table_name}")

    count = spark.read.format("delta").load(path).count()
    logger.info(f"[Gold] {table_name}: {count:,} rows → {path}")
    return count


def add_audit(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("_gold_created_at", F.current_timestamp())
        .withColumn("_run_date",         F.lit(RUN_DATE))
    )

In [0]:
logger.info("Building comment_sentiment ")

# Clean comment text once (mirrors Snowflake silver_comments LOWER + REGEXP)
df_comments_clean = (
    df_silver_comments
    .withColumn(
        "clean_comment",
        F.lower(
            F.regexp_replace(
                F.regexp_replace(F.trim(F.col("comment_text")), r"<[^>]+>", " "),
                r"\s+", " "
            )
        )
    )
    .filter(
        F.col("comment_text").isNotNull()
        & (F.length(F.trim(F.col("comment_text"))) >= 3)
    )
)

df_comment_scored = (
    df_comments_clean
    .withColumn("sentiment_score",   udf_sentiment_score(F.col("clean_comment")))
    .withColumn("sentiment_label",   udf_sentiment_label(F.col("sentiment_score")))
    .withColumn("emotion",           udf_emotion(F.col("clean_comment")))
    .withColumn("engagement_weight", F.log1p(F.col("like_count").cast("double")))
    .withColumn("pos_pattern",       F.lit("|".join(sorted(POS_WORDS))))
    .withColumn("neg_pattern",       F.lit("|".join(sorted(NEG_WORDS))))
    .withColumn(
        "positive_hits",
        F.size(F.expr(
            "filter(split(clean_comment, ' '), "
            f"x -> x rlike '({'|'.join(sorted(POS_WORDS))})')"
        ))
    )
    .withColumn(
        "negative_hits",
        F.size(F.expr(
            "filter(split(clean_comment, ' '), "
            f"x -> x rlike '({'|'.join(sorted(NEG_WORDS))})')"
        ))
    )
    .drop("pos_pattern", "neg_pattern")
    .select(
        "comment_id",
        "video_id",
        "author",
        "comment_text",
        "clean_comment",
        "like_count",
        "comment_date",
        "run_date",
        "word_count",
        "char_count",
        "has_emoji",
        "positive_hits",
        "negative_hits",
        "sentiment_score",
        "sentiment_label",
        "emotion",
        "engagement_weight",
    )
)

df_comment_scored = add_audit(df_comment_scored)
cnt_comments = write_sentiment_table(
    df_comment_scored,
    "comment_sentiment",
    partition_cols=["run_date"]
)

print(f"comment_sentiment: {cnt_comments:,} rows")

df_comment_scored.groupBy("sentiment_label").count().orderBy("sentiment_label").show()
df_comment_scored.groupBy("emotion").count().orderBy(F.col("count").desc()).show()

comment_sentiment: 22,584 rows
+---------------+-----+
|sentiment_label|count|
+---------------+-----+
|       NEGATIVE|  555|
|        NEUTRAL|17314|
|       POSITIVE| 4715|
+---------------+-----+

+--------+-----+
| emotion|count|
+--------+-----+
| NEUTRAL|19122|
|     JOY| 2331|
|   ANGER|  448|
| SADNESS|  347|
|SURPRISE|  226|
|    FEAR|   81|
| DISGUST|   29|
+--------+-----+



In [0]:
logger.info("Building video_sentiment_summary")

df_comment_agg = (
    df_comment_scored
    .groupBy("video_id", "run_date")
    .agg(
        F.count("comment_id")                                                   .alias("total_comments"),
        F.round(F.avg("sentiment_score"), 4)                                    .alias("avg_comment_sentiment"),
        F.round(F.percentile_approx("sentiment_score", 0.5), 4)                .alias("median_comment_sentiment"),
        F.round(F.stddev("sentiment_score"), 4)                                 .alias("stddev_comment_sentiment"),
        F.round(
            F.sum(F.col("sentiment_score") * F.col("engagement_weight"))
            / F.nullif(F.sum("engagement_weight"), F.lit(0)), 4
        )                                                                        .alias("weighted_avg_sentiment"),
        F.sum(F.when(F.col("sentiment_label") == "POSITIVE", 1).otherwise(0))  .alias("positive_comments"),
        F.sum(F.when(F.col("sentiment_label") == "NEGATIVE", 1).otherwise(0))  .alias("negative_comments"),
        F.sum(F.when(F.col("sentiment_label") == "NEUTRAL",  1).otherwise(0))  .alias("neutral_comments"),
        F.round(
            F.sum(F.when(F.col("sentiment_label") == "POSITIVE", 1).otherwise(0))
            / F.count("comment_id") * 100, 2
        )                                                                        .alias("pct_positive"),
        F.round(
            F.sum(F.when(F.col("sentiment_label") == "NEGATIVE", 1).otherwise(0))
            / F.count("comment_id") * 100, 2
        )                                                                        .alias("pct_negative"),
        F.round(
            F.sum(F.when(F.col("sentiment_label") == "NEUTRAL",  1).otherwise(0))
            / F.count("comment_id") * 100, 2
        )                                                                        .alias("pct_neutral"),
    
        F.expr("mode(emotion)")                                                  .alias("dominant_emotion"),
        
        F.sum("like_count")                                                      .alias("total_comment_likes"),
    )
)

df_title_scores = (
    df_silver_videos
    .select("video_id", "title", "channel_id", "channel_title", "category_id", "region_code")
    .withColumn("title_lower",          F.lower(F.col("title")))
    .withColumn("title_sentiment_score", udf_sentiment_score(F.col("title_lower")))
    .withColumn("title_sentiment_label", udf_sentiment_label(F.col("title_sentiment_score")))
    .withColumn("title_emotion",         udf_emotion(F.col("title_lower")))
    .drop("title_lower")
)

df_summary = (
    df_comment_agg
    .join(df_title_scores, on="video_id", how="left")
    .withColumn(
        "composite_sentiment_score",
        F.round(
            F.coalesce(F.col("avg_comment_sentiment"), F.lit(0.0)) * 0.80
            + F.coalesce(F.col("title_sentiment_score"),  F.lit(0.0)) * 0.20
        , 4)
    )
    .withColumn(
        "composite_sentiment_label",
        udf_sentiment_label(F.col("composite_sentiment_score"))
    )
    .select(
        "video_id", "run_date",
     
        "title", "channel_id", "channel_title", "category_id", "region_code",
    
        "total_comments", "total_comment_likes",
        "avg_comment_sentiment", "median_comment_sentiment",
        "stddev_comment_sentiment", "weighted_avg_sentiment",
        "positive_comments", "negative_comments", "neutral_comments",
        "pct_positive", "pct_negative", "pct_neutral",
        "dominant_emotion",
  
        "title_sentiment_score", "title_sentiment_label", "title_emotion",
     
        "composite_sentiment_score", "composite_sentiment_label",
    )
)

df_summary = add_audit(df_summary)
cnt_summary = write_sentiment_table(
    df_summary,
    "video_sentiment_summary",
    partition_cols=["region_code"]
)

print(f"video_sentiment_summary: {cnt_summary:,} rows")


df_summary.groupBy("composite_sentiment_label") \
    .agg(F.count("*").alias("videos"),
         F.round(F.avg("composite_sentiment_score"), 4).alias("avg_score")) \
    .orderBy("composite_sentiment_label") \
    .show()

video_sentiment_summary: 555 rows
+-------------------------+------+---------+
|composite_sentiment_label|videos|avg_score|
+-------------------------+------+---------+
|                 NEGATIVE|     3|  -0.1772|
|                  NEUTRAL|   328|   0.0596|
|                 POSITIVE|   224|   0.2773|
+-------------------------+------+---------+



In [0]:
logger.info("Building sentiment_keywords ")

# Capture for UDF closures (serverless does not support sparkContext.broadcast)
_stopwords = STOPWORDS
_min_len   = MIN_WORD_LEN


@F.udf(returnType=StringType())
def filter_keyword(word: str) -> str:
    if word is None:
        return None
    w = word.strip().lower()
    if len(w) < _min_len:
        return None
    if w in _stopwords:
        return None
    # Remove words that are pure numbers
    if re.fullmatch(r"\d+", w):
        return None
    return w


# Explode clean_comment into individual words per row
df_words = (
    df_comment_scored
    .select("video_id", "sentiment_label", "clean_comment")
    .withColumn("word", F.explode(F.split(F.col("clean_comment"), r"\s+")))
    .withColumn("word_clean", filter_keyword(F.col("word")))
    .filter(F.col("word_clean").isNotNull())
    .drop("word", "clean_comment")
)

# Count word frequency per video × sentiment bucket
df_keyword_counts = (
    df_words
    .groupBy("video_id", "sentiment_label", "word_clean")
    .agg(F.count("*").alias("frequency"))
)

# Rank within each video × sentiment and keep top N
window_kw = Window.partitionBy("video_id", "sentiment_label") \
                  .orderBy(F.col("frequency").desc())

df_keywords = (
    df_keyword_counts
    .withColumn("rank_within_sentiment", F.row_number().over(window_kw))
    .filter(F.col("rank_within_sentiment") <= TOP_KEYWORDS)
    .withColumnRenamed("word_clean", "keyword")
    .withColumn("run_date", F.lit(RUN_DATE))
)

df_keywords = add_audit(df_keywords)
cnt_keywords = write_sentiment_table(
    df_keywords,
    "sentiment_keywords",
    partition_cols=["sentiment_label"]
)

print(f"sentiment_keywords: {cnt_keywords:,} rows")

print("\n=== TOP POSITIVE KEYWORDS ===")
df_keywords.filter(F.col("sentiment_label") == "POSITIVE") \
    .groupBy("keyword").agg(F.sum("frequency").alias("total")) \
    .orderBy(F.col("total").desc()).show(15, truncate=False)

print("\n=== TOP NEGATIVE KEYWORDS ===")
df_keywords.filter(F.col("sentiment_label") == "NEGATIVE") \
    .groupBy("keyword").agg(F.sum("frequency").alias("total")) \
    .orderBy(F.col("total").desc()).show(15, truncate=False)

sentiment_keywords: 22,838 rows

=== TOP POSITIVE KEYWORDS ===
+-------+-----+
|keyword|total|
+-------+-----+
|the    |3804 |
|and    |2074 |
|you    |1384 |
|love   |943  |
|for    |804  |
|was    |424  |
|but    |372  |
|are    |306  |
|game   |226  |
|best   |225  |
|all    |223  |
|song   |220  |
|not    |203  |
|video  |191  |
|can    |183  |
+-------+-----+
only showing top 15 rows

=== TOP NEGATIVE KEYWORDS ===
+-------+-----+
|keyword|total|
+-------+-----+
|the    |446  |
|and    |221  |
|bacon  |203  |
|you    |96   |
|for    |95   |
|was    |80   |
|bad    |71   |
|not    |64   |
|hard   |53   |
|but    |47   |
|are    |33   |
|game   |29   |
|hate   |27   |
|his    |26   |
|had    |25   |
+-------+-----+
only showing top 15 rows


%md ## 9 · Data Quality Checks

In [0]:
def run_dq(path: str, name: str, not_null_cols: list, score_col: str = None):
    df    = spark.read.format("delta").load(path)
    total = df.count()
    logger.info(f"[DQ] {name}: {total:,} rows")

    for col in not_null_cols:
        null_n = df.filter(F.col(col).isNull()).count()
        status = "✓" if null_n == 0 else f"⚠ {null_n:,} NULLs"
        logger.info(f"  [not_null] {col}: {status}")

    if score_col:
        out_of_range = df.filter(
            (F.col(score_col) < -1.0) | (F.col(score_col) > 1.0)
        ).count()
        status = "✓" if out_of_range == 0 else f"⚠ {out_of_range} out of [-1,+1]"
        logger.info(f"  [score_range] {score_col}: {status}")


run_dq(
    f"{SENTIMENT_PATH}/comment_sentiment",
    "comment_sentiment",
    ["comment_id", "video_id", "sentiment_label", "emotion"],
    "sentiment_score"
)

run_dq(
    f"{SENTIMENT_PATH}/video_sentiment_summary",
    "video_sentiment_summary",
    ["video_id", "run_date", "composite_sentiment_label"],
    "composite_sentiment_score"
)

run_dq(
    f"{SENTIMENT_PATH}/sentiment_keywords",
    "sentiment_keywords",
    ["video_id", "sentiment_label", "keyword"],
)

%md ## 10 · Checkpoint

In [0]:
summary = {
    "layer":                          "gold_sentiment",
    "run_date":                       RUN_DATE,
    "processed_at":                   datetime.now(timezone.utc).isoformat(),
    "comment_sentiment_rows":    cnt_comments,
    "video_summary_rows":        cnt_summary,
    "sentiment_keywords_rows":   cnt_keywords,
}



%md ## 11 · Summary

In [0]:
print("=" * 65)
print("  SENTIMENT ANALYSIS — COMPLETE SUMMARY")
print("=" * 65)
print(f"  Run Date                          : {RUN_DATE}")
print()
print("  GOLD TABLES WRITTEN")
print(f"  gold_comment_sentiment            : {cnt_comments:>8,} rows")
print(f"  gold_video_sentiment_summary      : {cnt_summary:>8,} rows")
print(f"  gold_sentiment_keywords           : {cnt_keywords:>8,} rows")
print()
print("  SENTIMENT LOGIC")
print(f"  Positive words                    : {len(POS_WORDS):>8}")
print(f"  Negative words                    : {len(NEG_WORDS):>8}")
print(f"  Intensifier words                 : {len(INTENSIFIERS):>8}")
print(f"  Negation words                    : {len(NEGATIONS):>8}")
print(f"  Emotion categories                : {len(EMOTION_MAP):>8}")
print(f"  Top keywords per bucket           : {TOP_KEYWORDS:>8}")
print()
print("  COMPOSITE FORMULA")
print("  = comment_score × 0.80 + title_udf_score × 0.20")
print("  (mirrors Snowflake VW_SENTIMENT_ENHANCED exactly)")
print("=" * 65)

dbutils.notebook.exit(json.dumps(summary))

In [0]:
%sql
select distinct region_code from `youtube-capstone-catalog`.bronze.bronze_videos;

region_code
US
GB
AU
IN
CA
